### SPRINT 0 - Base de l'Agent Médical et Prétraitement
#### **Objectif:** Créer la structure de base de l'agent médical et le prétraitement des données médicales


##### Installation des bibliothèques nécessaires pour le traitement de texte médical

In [1]:
!pip install pandas numpy scikit-learn nltk

Import des bibliothèques Python nécessaires pour le prétraitement

In [2]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer


Téléchargement des données nécessaires pour le tokenizing et les stopwords

In [7]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True


### Classe de prétraitement des textes médicaux
#### Cette classe effectue:
##### - Nettoyage du texte
##### - Tokenisation
##### - Suppression des stopwords généraux et médicaux
##### - Stemming (réduction des mots à leur racine)

In [4]:
class PreprocesseurMedical:
    def __init__(self):
        self.stemmer = SnowballStemmer("french")
        self.stop_words = set(stopwords.words('french'))
        # Ajout de stopwords médicaux spécifiques
        self.stop_words_medicaux = {
            'patient', 'docteur', 'médecin', 'symptôme', 'symptômes',
            'traitement', 'maladie', 'diagnostic', 'consultation'
        }
        self.stop_words.update(self.stop_words_medicaux)

    def nettoyer_texte(self, texte):
        """Nettoyer le texte médical en minuscules et supprimer caractères spéciaux"""
        if pd.isna(texte):
            return ""

        texte = str(texte).lower()
        texte = re.sub(r'[^a-zàâäéèêëîïôöùûüç\s]', ' ', texte)
        texte = re.sub(r'\s+', ' ', texte).strip()
        return texte

    def tokeniser(self, texte):
        """Tokeniser le texte en mots"""
        return word_tokenize(texte, language='french')

    def supprimer_stopwords(self, tokens):
        """Supprimer les mots vides et très courts"""
        return [token for token in tokens if token not in self.stop_words and len(token) > 2]

    def stemmer_tokens(self, tokens):
        """Réduire les mots à leur racine (stemming)"""
        return [self.stemmer.stem(token) for token in tokens]

    def preprocesser_texte(self, texte):
        """Pipeline complet de prétraitement"""
        texte_nettoye = self.nettoyer_texte(texte)
        tokens = self.tokeniser(texte_nettoye)
        tokens_sans_stopwords = self.supprimer_stopwords(tokens)
        tokens_stemmes = self.stemmer_tokens(tokens_sans_stopwords)

        return {
            'texte_original': texte,
            'texte_nettoye': texte_nettoye,
            'tokens': tokens_stemmes,
            'texte_final': ' '.join(tokens_stemmes)
        }


### Création d'une base de données médicale simulée
#### Contient des symptômes, diagnostics et spécialités correspondantes pour tests


In [5]:
def creer_base_donnees_medicale():
    """Créer une base de données médicale de démonstration"""
    data = {
        'symptomes': [
            "fièvre toux fatigue maux de tête",
            "douleur abdominale nausées vomissements",
            "douleur thoracique essoufflement palpitations",
            "maux de gorge écoulement nasal éternuements",
            "douleur articulaire gonflement rougeur",
            "vertiges nausées vision floue",
            "douleur dos difficulté mouvement raideur",
            "fatigue extrême pâleur essoufflement",
            "brûlures urinaire fréquent envie uriner",
            "éruption cutanée démangeaisons rougeur peau"
        ],
        'diagnostics': [
            "grippe",
            "gastro-entérite",
            "problème cardiaque",
            "rhume",
            "arthrite",
            "hypotension",
            "lombalgie",
            "anémie",
            "infection urinaire",
            "allergie cutanée"
        ],
        'specialites': [
            "médecine générale",
            "gastro-entérologie",
            "cardiologie",
            "médecine générale",
            "rhumatologie",
            "médecine générale",
            "orthopédie",
            "hématologie",
            "urologie",
            "dermatologie"
        ]
    }
    return pd.DataFrame(data)


Appliquer la classe de prétraitement sur la base de données médicale simulée

In [8]:
print("=== SPRINT 0 - TEST PRÉTRAITEMENT ===")
preprocesseur = PreprocesseurMedical()
df_medical = creer_base_donnees_medicale()

# Appliquer le prétraitement sur chaque ligne
resultats_preprocess = []
for symptome in df_medical['symptomes']:
    resultat = preprocesseur.preprocesser_texte(symptome)
    resultats_preprocess.append(resultat)

# Afficher les 3 premiers résultats pour vérification
for i, resultat in enumerate(resultats_preprocess[:3]):
    print(f"\nExemple {i+1}:")
    print(f"Original: {resultat['texte_original']}")
    print(f"Nettoyé: {resultat['texte_nettoye']}")
    print(f"Tokens: {resultat['tokens']}")

print(f"\nBase de données créée avec {len(df_medical)} entrées médicales")


=== SPRINT 1 - TEST PRÉTRAITEMENT ===

Exemple 1:
Original: fièvre toux fatigue maux de tête
Nettoyé: fièvre toux fatigue maux de tête
Tokens: ['fievr', 'toux', 'fatigu', 'maux', 'têt']

Exemple 2:
Original: douleur abdominale nausées vomissements
Nettoyé: douleur abdominale nausées vomissements
Tokens: ['douleur', 'abdominal', 'naus', 'vom']

Exemple 3:
Original: douleur thoracique essoufflement palpitations
Nettoyé: douleur thoracique essoufflement palpitations
Tokens: ['douleur', 'thorac', 'essouffl', 'palpit']

Base de données créée avec 10 entrées médicales


### SPRINT 1 - Analyse de Similarité et Recherche de Symptômes
##### **Objectif:** Implémenter la recherche par similarité et la classification basique des symptômes

In [9]:
!pip install scipy scikit-learn

TF-IDF pour vectoriser les symptômes, Cosine similarity pour la recherche par similarité

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
import numpy as np


### Classe AnalyseurSymptomes
#### Cette classe effectue :
##### - Prétraitement des symptômes
##### - Vectorisation TF-IDF
##### - Recherche par similarité cosinus
##### - Analyse basique du niveau d'urgence

In [11]:
class AnalyseurSymptomes:
    def __init__(self, df_medical):
        self.df = df_medical
        self.vectorizer = TfidfVectorizer()
        self.preprocesseur = PreprocesseurMedical()
        self.encoder_specialites = LabelEncoder()
        self.encoder_diagnostics = LabelEncoder()

        self.preparer_donnees()

    def preparer_donnees(self):
        """Prétraiter et vectoriser les symptômes, encoder les spécialités et diagnostics"""
        symptomes_preprocesses = []
        for symptome in self.df['symptomes']:
            resultat = self.preprocesseur.preprocesser_texte(symptome)
            symptomes_preprocesses.append(resultat['texte_final'])

        self.symptomes_processed = symptomes_preprocesses
        self.matrice_tfidf = self.vectorizer.fit_transform(self.symptomes_processed)

        self.specialites_encoded = self.encoder_specialites.fit_transform(self.df['specialites'])
        self.diagnostics_encoded = self.encoder_diagnostics.fit_transform(self.df['diagnostics'])

    def rechercher_similarite(self, symptomes_utilisateur, top_n=3):
        """Rechercher les diagnostics les plus similaires aux symptômes donnés"""
        resultat = self.preprocesseur.preprocesser_texte(symptomes_utilisateur)
        symptomes_vector = self.vectorizer.transform([resultat['texte_final']])

        similarites = cosine_similarity(symptomes_vector, self.matrice_tfidf)
        indices_similaires = similarites.argsort()[0][-top_n:][::-1]

        resultats = []
        for idx in indices_similaires:
            similarite = similarites[0][idx]
            if similarite > 0.1:
                resultats.append({
                    'symptomes_similaires': self.df.iloc[idx]['symptomes'],
                    'diagnostic_suggere': self.df.iloc[idx]['diagnostics'],
                    'specialite_suggeree': self.df.iloc[idx]['specialites'],
                    'score_similarite': round(similarite, 3)
                })

        return resultats

    def analyser_symptomes(self, symptomes_utilisateur):
        """Analyser les symptômes de l'utilisateur et fournir recommandations et diagnostics"""
        resultats_similarite = self.rechercher_similarite(symptomes_utilisateur)

        if not resultats_similarite:
            return {
                'urgence': 'inconnue',
                'recommandation': 'Consultez un médecin généraliste pour une évaluation',
                'diagnostics_possibles': []
            }

        symptomes_lower = symptomes_utilisateur.lower()
        urgences_mots_cles = {
            'douleur thoracique': 'élevée',
            'essoufflement': 'élevée',
            'saignement abondant': 'élevée',
            'perte connaissance': 'élevée',
            'fièvre élevée': 'modérée',
            'vomissements persistants': 'modérée',
            'douleur intense': 'modérée'
        }

        urgence = 'faible'
        for mot_cle, niveau in urgences_mots_cles.items():
            if mot_cle in symptomes_lower:
                urgence = niveau
                break

        return {
            'urgence': urgence,
            'recommandation': f"Consultez un {resultats_similarite[0]['specialite_suggeree']}",
            'diagnostics_possibles': resultats_similarite
        }


Tester la recherche par similarité et l'analyse de niveau d'urgence

In [12]:
print("\n=== SPRINT 1 - TEST ANALYSE SYMPTÔMES ===")
analyseur = AnalyseurSymptomes(df_medical)

# Liste de symptômes à tester
symptomes_tests = [
    "j'ai de la fièvre et je tousse beaucoup",
    "douleur dans la poitrine et essoufflement",
    "mal au ventre avec nausées"
]

# Analyse de chaque exemple
for symptomes in symptomes_tests:
    print(f"\n--- Analyse pour: '{symptomes}' ---")
    resultat = analyseur.analyser_symptomes(symptomes)
    print(f"Urgence: {resultat['urgence']}")
    print(f"Recommandation: {resultat['recommandation']}")
    print("Diagnostics possibles:")
    for diag in resultat['diagnostics_possibles']:
        print(f"  - {diag['diagnostic_suggere']} (score: {diag['score_similarite']})")



=== SPRINT 2 - TEST ANALYSE SYMPTÔMES ===

--- Analyse pour: 'j'ai de la fièvre et je tousse beaucoup' ---
Urgence: faible
Recommandation: Consultez un médecine générale
Diagnostics possibles:
  - grippe (score: 0.474)

--- Analyse pour: 'douleur dans la poitrine et essoufflement' ---
Urgence: élevée
Recommandation: Consultez un cardiologie
Diagnostics possibles:
  - problème cardiaque (score: 0.606)
  - anémie (score: 0.362)
  - gastro-entérite (score: 0.228)

--- Analyse pour: 'mal au ventre avec nausées' ---
Urgence: faible
Recommandation: Consultez un gastro-entérologie
Diagnostics possibles:
  - gastro-entérite (score: 0.478)
  - hypotension (score: 0.441)
